In [ ]:
import pandas as pd
import geopandas as gpd

In [44]:
# 데이터 로드
csv_path = 'C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/인구/주민등록인구기타현황(고령 인구현황)_월간.csv'
elderly = pd.read_csv(csv_path, encoding='cp949', thousands=',')
elderly.head()

,행정구역,2026년04월_전체,2026년04월_남자,2026년04월_여자,2026년04월_65세이상전체,2026년04월_65세이상남자,2026년04월_65세이상여자
0,서울특별시 (1100000000),9298673,4477462,4821211,1934596,855406,1079190
1,서울특별시 종로구 (1111000000),136716,65420,71296,31139,14015,17124
2,서울특별시 종로구 청운효자동(1111051500),10773,4880,5893,2308,951,1357
3,서울특별시 종로구 사직동(1111053000),8876,3915,4961,2000,857,1143
4,서울특별시 종로구 삼청동(1111054000),2078,986,1092,607,261,346


In [45]:
elderly = elderly[['행정구역', '2026년04월_전체', '2026년04월_65세이상전체']]

In [46]:
# 서울특별시 행만 출력
elderly = elderly[elderly['행정구역'].str.contains('서울특별시', na = False)]
elderly.shape

(453, 3)

In [47]:
condition = elderly['행정구역'].str.split('(').str[0].str.strip().str.endswith('동', na=False)
elderly = elderly[condition]

elderly.head()

,행정구역,2026년04월_전체,2026년04월_65세이상전체
2,서울특별시 종로구 청운효자동(1111051500),10773,2308
3,서울특별시 종로구 사직동(1111053000),8876,2000
4,서울특별시 종로구 삼청동(1111054000),2078,607
5,서울특별시 종로구 부암동(1111055000),8874,2052
6,서울특별시 종로구 평창동(1111056000),16973,4119


In [48]:
elderly['행정구역코드'] = elderly['행정구역'].str.extract(r'\((\d+)\)')
elderly['행정구역'] = elderly['행정구역'].str.split('(').str[0].str.strip().str.split().str[-1]

elderly.head()

,행정구역,2026년04월_전체,2026년04월_65세이상전체,행정구역코드
2,청운효자동,10773,2308,1111051500
3,사직동,8876,2000,1111053000
4,삼청동,2078,607,1111054000
5,부암동,8874,2052,1111055000
6,평창동,16973,4119,1111056000


In [49]:
# 노인인구비율 파생변수 생성
elderly['노인인구비율'] = elderly['2026년04월_65세이상전체'] / elderly['2026년04월_전체']
elderly.head()

,행정구역,2026년04월_전체,2026년04월_65세이상전체,행정구역코드,노인인구비율
2,청운효자동,10773,2308,1111051500,0.214239
3,사직동,8876,2000,1111053000,0.225327
4,삼청동,2078,607,1111054000,0.292108
5,부암동,8874,2052,1111055000,0.231237
6,평창동,16973,4119,1111056000,0.242680


In [50]:
# 지도 데이터 불러오기
gdf = gpd.read_file("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/행정구역/행정구역_행정동_경계/BND_ADM_DONG_PG.shp", encoding='cp949')

In [ ]:
# 지도의 'ADM_NM'과 인구데이터의 '행정구역'을 기준으로 병합
merged_elderly_gdf = gdf.merge(elderly, left_on = 'ADM_NM', right_on = '행정구역', how = 'left')

# 데이터가 없어 NaN이 된 비율은 0으로 채움
merged_elderly_gdf['노인인구비율'] = merged_elderly_gdf['노인인구비율'].fillna(0)

In [56]:
# 지도 시각화
m_elderly = merged_elderly_gdf.explore(
    column = '노인인구비율',        
    cmap = 'YlOrRd',                       
    tooltip = ['ADM_NM', '행정구역', '노인인구비율'],  
    style_kwds = {'weight': 0.5, 'color': 'gray', 'fillOpacity': 0.8}
)

In [57]:
m_elderly.save("seoul_elderly_ratio_map.html")